# Pixels to Predictions — SmolVLM-500M MCQA

End-to-end notebook for the DL Vision Challenge. We fine-tune **HuggingFaceTB/SmolVLM-500M-Instruct** with a tiny LoRA adapter (well under the 5M trainable-parameter cap) and answer each multiple-choice question by scoring the *letter* tokens (A/B/C/D/E) under the model's vision-language head. This handles the variable choice count (2–5) cleanly and needs no extra classifier weights.

**Pipeline**
1. Load CSVs and resolve image paths.
2. Build prompts: question + hint + lecture + lettered choices, with the image.
3. **Train** (one short epoch with LoRA on the language model's q/k/v/o projections only) — supervised on the correct letter token.
4. **Predict** by scoring each choice's letter token log-probability and taking argmax over the *valid* letters for that question.
5. Write `submission.csv`.

Designed to run on a single Kaggle T4/P100 (or Colab T4) within free-tier limits. If you want a zero-training baseline (still strong), set `DO_TRAIN = False` in the config cell.

## 1. Configuration

Edit the paths if your Kaggle dataset name differs. The defaults assume the competition data is attached at `/kaggle/input/pixels-to-predictions/` with the layout described in the task.

In [ ]:
import os, sys, json, math, random, gc, time
from pathlib import Path

# --- Paths ----------------------------------------------------------------
# On Kaggle the competition data is typically under /kaggle/input/<competition-name>/
# Adjust DATA_ROOT if your attached dataset has a different folder name.
CANDIDATE_ROOTS = [
    "/kaggle/input/pixels-to-predictions",
    "/kaggle/input/pixels-to-predictions-dl-vision-challenge",
    "/content/pixels-to-predictions",
    "./pixels-to-predictions",
    ".",
]
DATA_ROOT = next((p for p in CANDIDATE_ROOTS if os.path.exists(os.path.join(p, "train.csv"))), None)
assert DATA_ROOT is not None, f"Could not find train.csv under any of {CANDIDATE_ROOTS}. Set DATA_ROOT manually."
print("DATA_ROOT =", DATA_ROOT)

# Image paths in the CSV are like "images/train/train_xxxxx.png".
# In some packagings the actual files live at "<root>/images/<split>/..." (one level)
# and in others at "<root>/images/images/<split>/..." (nested). We auto-detect.
def resolve_image_root(data_root):
    candidates = [
        os.path.join(data_root, "images", "train"),                # CSV path "images/train/x.png" -> data_root + path
        os.path.join(data_root, "images", "images", "train"),      # CSV path resolves under data_root/images/
    ]
    if os.path.exists(candidates[0]):
        return data_root                                            # prefix = data_root
    if os.path.exists(candidates[1]):
        return os.path.join(data_root, "images")                   # prefix = data_root/images
    raise FileNotFoundError("Cannot locate image directories.")
IMAGE_PREFIX = resolve_image_root(DATA_ROOT)
print("IMAGE_PREFIX =", IMAGE_PREFIX)

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Model ----------------------------------------------------------------
MODEL_ID   = "HuggingFaceTB/SmolVLM-500M-Instruct"
MAX_IMG_SIZE = 384                  # downsize images for speed/memory
MAX_TEXT_CHARS = 1500                # truncate very long lecture/hint text

# --- Training -------------------------------------------------------------
DO_TRAIN          = True             # set False for a zero-training baseline
NUM_EPOCHS        = 1
BATCH_SIZE        = 1                # SmolVLM-500M with images fits ~1 per step on T4
GRAD_ACCUM_STEPS  = 8                # effective batch size 8
LR                = 2e-4
WEIGHT_DECAY      = 0.0
WARMUP_RATIO      = 0.03
MAX_TRAIN_SAMPLES = None             # set to e.g. 1500 for a quick smoke run
SEED              = 42

# --- LoRA (keeps trainable params well below the 5M cap) -----------------
LORA_R            = 8
LORA_ALPHA        = 16
LORA_DROPOUT      = 0.05
LORA_TARGETS      = ["q_proj", "k_proj", "v_proj", "o_proj"]   # LM attention only

# --- Inference ------------------------------------------------------------
EVAL_BATCH_SIZE   = 1
EVAL_VAL          = True             # evaluate on val.csv after training

random.seed(SEED)
import numpy as np; np.random.seed(SEED)
print("Config loaded.")

## 2. Install / verify dependencies

Kaggle's default image already has `torch`, `transformers`, `pillow`, `pandas`. We pin minimum versions that include SmolVLM support and add `peft` for LoRA. The `pip install` runs only if a version is missing or too old, so re-running the notebook is fast.

In [ ]:
import importlib, subprocess, sys
def ensure(pkg, import_name=None, min_version=None):
    name = import_name or pkg.split("==")[0].split(">=")[0]
    try:
        m = importlib.import_module(name)
        if min_version:
            from packaging.version import Version
            if Version(getattr(m, "__version__", "0")) < Version(min_version):
                raise ImportError
        return
    except Exception:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

ensure("transformers>=4.46.0", "transformers", "4.46.0")
ensure("peft>=0.13.0", "peft", "0.13.0")
ensure("accelerate>=0.34.0", "accelerate", "0.34.0")
ensure("pillow", "PIL")
ensure("pandas")
ensure("numpy")

import torch, transformers, peft
print("torch:", torch.__version__, "| transformers:", transformers.__version__, "| peft:", peft.__version__)
print("CUDA available:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## 3. Load CSVs and clean text

We keep the question, hint, and lecture text but truncate aggressively so the prompt fits comfortably in context.

In [ ]:
import pandas as pd, json
import numpy as np

train_df = pd.read_csv(os.path.join(DATA_ROOT, "train.csv"))
val_df   = pd.read_csv(os.path.join(DATA_ROOT, "val.csv"))
test_df  = pd.read_csv(os.path.join(DATA_ROOT, "test.csv"))

print(f"train: {len(train_df)} | val: {len(val_df)} | test: {len(test_df)}")

def parse_choices(s):
    """choices column is JSON-encoded; fall back to a forgiving parse."""
    if isinstance(s, list): return s
    try:    return json.loads(s)
    except Exception:
        try: return json.loads(s.replace("\\", "\\\\"))
        except Exception:
            return [c.strip() for c in str(s).strip("[]").split(",")]

for df in (train_df, val_df, test_df):
    df["choices_list"] = df["choices"].apply(parse_choices)
    # Defensive: re-derive num_choices in case it disagrees with the parsed list
    df["num_choices"] = df["choices_list"].apply(len)

print("num_choices distribution (train):", train_df.num_choices.value_counts().to_dict())
if "answer" in train_df.columns:
    print("answer distribution (train):", train_df.answer.value_counts().sort_index().to_dict())

def clean(t):
    if t is None or (isinstance(t, float) and math.isnan(t)): return ""
    t = str(t).strip()
    if len(t) > MAX_TEXT_CHARS:
        t = t[:MAX_TEXT_CHARS] + " ..."
    return t

train_df["hint_c"]    = train_df["hint"].apply(clean)
train_df["lecture_c"] = train_df["lecture"].apply(clean)
val_df["hint_c"]      = val_df["hint"].apply(clean)
val_df["lecture_c"]   = val_df["lecture"].apply(clean)
test_df["hint_c"]     = test_df["hint"].apply(clean)
test_df["lecture_c"]  = test_df["lecture"].apply(clean)

# Resolve image path on disk
def resolve_path(rel_path):
    # CSV gives "images/<split>/<file>". Prefix is either DATA_ROOT or DATA_ROOT/images.
    p = os.path.join(IMAGE_PREFIX, rel_path)
    if os.path.exists(p): return p
    # Fallback: maybe IMAGE_PREFIX already includes "images"
    p2 = os.path.join(IMAGE_PREFIX, rel_path.split("/", 1)[-1])
    if os.path.exists(p2): return p2
    return None

for df in (train_df, val_df, test_df):
    df["image_full"] = df["image_path"].apply(resolve_path)
missing = [df["image_full"].isna().sum() for df in (train_df, val_df, test_df)]
print("Missing images train/val/test:", missing)
assert sum(missing) == 0, "Some images could not be located. Check IMAGE_PREFIX."

## 4. Load the SmolVLM-500M-Instruct model and processor

We load in fp16 to fit comfortably on a T4. The processor handles image preprocessing and chat-formatted prompts.

In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if device == "cuda" else torch.float32

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
    _attn_implementation="eager",   # safer on T4; flash-attn isn\'t available
).to(device)

# Total param count
total_params = sum(p.numel() for p in model.parameters())
print(f"Total SmolVLM params: {total_params/1e6:.1f}M (frozen baseline)")

# Identify the letter token IDs we will score against. SmolVLM's tokenizer is the
# llama-style tokenizer used by SmolLM; a leading space is significant.
tokenizer = processor.tokenizer
LETTERS = ["A", "B", "C", "D", "E"]
def letter_token_id(letter):
    # Tokenize " A" (with leading space, which is how it would appear after "Answer:")
    ids = tokenizer.encode(f" {letter}", add_special_tokens=False)
    # Use the LAST id so that BPE merges that prepend a space-marker still work.
    return ids[-1]
LETTER_IDS = [letter_token_id(l) for l in LETTERS]
print("Letter token ids:", dict(zip(LETTERS, LETTER_IDS)))
# Sanity: they must be distinct
assert len(set(LETTER_IDS)) == len(LETTER_IDS), "Letter token ids collide; check tokenizer."

## 5. Attach a small LoRA adapter

We freeze the entire base model and train a LoRA adapter on the language-model attention projections only. With `r=8` over 4 projection types in ~30 transformer layers this stays well under the 5M trainable-parameter cap.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# Freeze every base parameter
for p in model.parameters():
    p.requires_grad = False

if DO_TRAIN:
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=LORA_TARGETS,
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable params (LoRA): {trainable:,}  ({trainable/1e6:.3f}M)")
    assert trainable < 5_000_000, (
        f"Trainable params {trainable:,} exceed 5M cap. Reduce LORA_R or LORA_TARGETS."
    )
else:
    print("DO_TRAIN=False → using base model with no fine-tuning.")

## 6. Prompt construction

For each example we build a chat-formatted prompt:

```
<image>
Question: ...
Hint: ...        (omitted if empty)
Lecture: ...     (omitted if empty)
Choices:
A. ...
B. ...
...
Answer:
```

The model is then asked to produce a single letter. During training we supervise only the letter token; during inference we read the next-token logits over `{A, B, C, D, E}` and pick the argmax restricted to the letters that exist for this question.

In [ ]:
from PIL import Image

def build_user_text(row):
    """Plain text portion of the user message (image is added separately)."""
    q = str(row["question"]).strip()
    parts = [f"Question: {q}"]
    h = row.get("hint_c", "")
    if h: parts.append(f"Hint: {h}")
    L = row.get("lecture_c", "")
    if L: parts.append(f"Lecture: {L}")
    parts.append("Choices:")
    for i, c in enumerate(row["choices_list"]):
        parts.append(f"{LETTERS[i]}. {str(c).strip()}")
    parts.append("Answer with the single letter of the best choice.")
    return "\n".join(parts)

def load_image(path):
    img = Image.open(path).convert("RGB")
    # Downsize on the long side for speed
    w, h = img.size
    if max(w, h) > MAX_IMG_SIZE:
        if w >= h:
            nw, nh = MAX_IMG_SIZE, int(h * MAX_IMG_SIZE / w)
        else:
            nh, nw = MAX_IMG_SIZE, int(w * MAX_IMG_SIZE / h)
        img = img.resize((nw, nh), Image.BILINEAR)
    return img

def build_chat_messages(row):
    return [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": build_user_text(row)},
        ],
    }]

# Quick sanity check — render one prompt
sample = train_df.iloc[0]
msgs = build_chat_messages(sample)
prompt_text = processor.apply_chat_template(msgs, add_generation_prompt=True)
print(prompt_text[:600])
print("...")
print(f"\n[GROUND-TRUTH LETTER: {LETTERS[int(sample['answer'])]}]")

## 7. Datasets and collator

We build a PyTorch `Dataset` that emits `(image, prompt_text, target_letter_id)` tuples; the collator runs the processor and adds labels that mask out everything except the assistant's letter token.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class MCQADataset(Dataset):
    def __init__(self, df, with_labels=True):
        self.df = df.reset_index(drop=True)
        self.with_labels = with_labels and ("answer" in df.columns)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = load_image(row["image_full"])
        msgs  = build_chat_messages(row)
        prompt_text = processor.apply_chat_template(msgs, add_generation_prompt=True)
        item = {"image": image, "prompt_text": prompt_text, "row_idx": idx}
        if self.with_labels:
            ans = int(row["answer"])
            item["answer_idx"]    = ans
            item["letter_id"]     = LETTER_IDS[ans]
            item["num_choices"]   = int(row["num_choices"])
        else:
            item["num_choices"]   = int(row["num_choices"])
        return item

def collate_train(batch):
    """Build inputs for supervised training: prompt + " <LETTER>" with labels masking everything but the letter."""
    prompt_texts = []
    images       = []
    target_ids   = []
    for b in batch:
        # We append the gold letter to the prompt so the model learns to predict it as the next token.
        # The processor will tokenize prompt+answer; we then build labels that are -100 everywhere
        # except on the single answer-token position.
        full_text = b["prompt_text"] + LETTERS[b["answer_idx"]]
        prompt_texts.append(full_text)
        images.append([b["image"]])
        target_ids.append(b["letter_id"])

    enc = processor(text=prompt_texts, images=images, return_tensors="pt", padding=True)

    input_ids = enc["input_ids"]
    labels = torch.full_like(input_ids, -100)
    # The supervised position is the LAST non-pad token of each row (= the letter token we appended)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    for i in range(input_ids.size(0)):
        nonpad = (input_ids[i] != pad_id).nonzero(as_tuple=True)[0]
        last = nonpad[-1].item()
        # Confirm the last token actually equals the intended letter id; if not, fall back to it.
        labels[i, last] = input_ids[i, last].item()
    enc["labels"] = labels
    return enc

def collate_eval(batch):
    """Build inputs for inference: prompt only; we'll read next-token logits."""
    prompt_texts = [b["prompt_text"] for b in batch]
    images       = [[b["image"]] for b in batch]
    enc = processor(text=prompt_texts, images=images, return_tensors="pt", padding=True)
    enc["_num_choices"] = torch.tensor([b["num_choices"] for b in batch], dtype=torch.long)
    enc["_row_idx"]     = torch.tensor([b["row_idx"] for b in batch], dtype=torch.long)
    return enc

train_ds = MCQADataset(train_df, with_labels=True)
val_ds   = MCQADataset(val_df,   with_labels=True)
test_ds  = MCQADataset(test_df,  with_labels=False)

if MAX_TRAIN_SAMPLES is not None:
    train_ds = torch.utils.data.Subset(train_ds, list(range(min(MAX_TRAIN_SAMPLES, len(train_ds)))))

print(f"train_ds={len(train_ds)} val_ds={len(val_ds)} test_ds={len(test_ds)}")

## 8. Training loop

A single short epoch with gradient accumulation. We log loss every ~50 optimizer steps. This is a deliberately minimal trainer (no `Trainer` class) because the chat template + image batching with SmolVLM is cleaner to control directly.

In [ ]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

def train_one_epoch():
    model.train()
    loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=collate_train, num_workers=2, pin_memory=True,
    )
    num_steps = math.ceil(len(loader) / GRAD_ACCUM_STEPS) * NUM_EPOCHS
    optim = AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=WEIGHT_DECAY)
    sched = get_cosine_schedule_with_warmup(optim, int(WARMUP_RATIO * num_steps), num_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

    step = 0
    running = 0.0
    t0 = time.time()
    for epoch in range(NUM_EPOCHS):
        optim.zero_grad()
        for i, batch in enumerate(loader):
            batch = {k: v.to(device, non_blocking=True) if torch.is_tensor(v) else v for k, v in batch.items()}
            with torch.amp.autocast("cuda", dtype=dtype, enabled=(device == "cuda")):
                out = model(**batch)
                loss = out.loss / GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()
            running += loss.item() * GRAD_ACCUM_STEPS

            if (i + 1) % GRAD_ACCUM_STEPS == 0 or (i + 1) == len(loader):
                scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0,
                )
                scaler.step(optim)
                scaler.update()
                sched.step()
                optim.zero_grad()
                step += 1
                if step % 25 == 0:
                    seen = (i + 1)
                    avg = running / seen
                    elapsed = time.time() - t0
                    print(f"epoch {epoch+1} step {step:4d} | seen {seen}/{len(loader)} | avg loss {avg:.4f} | {elapsed:.0f}s")
        print(f"--- epoch {epoch+1} done | avg loss {running/len(loader):.4f} ---")
    return

if DO_TRAIN:
    train_one_epoch()
    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
else:
    print("Skipping training (DO_TRAIN=False).")

## 9. Inference: score the letter tokens

We feed the prompt (no answer appended) and read the logits at the final non-pad position. Among the letters valid for that example (`A` … letter for `num_choices-1`), we pick the highest-probability letter.

In [ ]:
@torch.no_grad()
def predict_indices(dataset, batch_size=EVAL_BATCH_SIZE, desc="predict"):
    model.eval()
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False,
        collate_fn=collate_eval, num_workers=2, pin_memory=True,
    )
    preds = [None] * len(dataset)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    letter_ids_t = torch.tensor(LETTER_IDS, device=device)

    t0 = time.time()
    n_done = 0
    for batch in loader:
        num_choices = batch.pop("_num_choices")
        row_idx     = batch.pop("_row_idx")
        batch = {k: v.to(device, non_blocking=True) if torch.is_tensor(v) else v for k, v in batch.items()}
        with torch.amp.autocast("cuda", dtype=dtype, enabled=(device == "cuda")):
            out = model(**batch)
        logits = out.logits  # [B, T, V]
        input_ids = batch["input_ids"]
        for b in range(input_ids.size(0)):
            nonpad = (input_ids[b] != pad_id).nonzero(as_tuple=True)[0]
            last = nonpad[-1].item()
            # Next-token logits live AT position `last` (predicting token last+1).
            row_logits = logits[b, last]
            letter_logits = row_logits[letter_ids_t]            # length 5
            nc = int(num_choices[b].item())
            letter_logits = letter_logits[:nc]                  # restrict to valid letters
            pred = int(letter_logits.argmax().item())
            preds[int(row_idx[b].item())] = pred
        n_done += input_ids.size(0)
        if n_done % 100 == 0:
            print(f"  {desc}: {n_done}/{len(dataset)} ({time.time()-t0:.0f}s)")
    return preds

## 10. Validation accuracy

In [ ]:
if EVAL_VAL:
    val_preds = predict_indices(val_ds, desc="val")
    y_true = val_df["answer"].astype(int).tolist()
    acc = float(np.mean([p == t for p, t in zip(val_preds, y_true)]))
    print(f"\nValidation accuracy: {acc:.4f}  ({sum(p==t for p,t in zip(val_preds,y_true))}/{len(y_true)})")
    # Per-num-choices breakdown
    val_df_eval = val_df.copy()
    val_df_eval["pred"] = val_preds
    print("\nAccuracy by num_choices:")
    print(val_df_eval.groupby("num_choices").apply(lambda d: (d["pred"]==d["answer"]).mean()).round(3))
else:
    print("Skipping validation.")

## 11. Test predictions and submission file

In [ ]:
test_preds = predict_indices(test_ds, desc="test")

submission = pd.DataFrame({
    "id": test_df["id"].tolist(),
    "answer": [int(p) for p in test_preds],
})

# Defensive: ensure every predicted index is valid for its question's choice set
nc = test_df["num_choices"].astype(int).tolist()
fixed = 0
for i, (a, k) in enumerate(zip(submission["answer"], nc)):
    if not (0 <= a < k):
        submission.at[i, "answer"] = 0
        fixed += 1
if fixed:
    print(f"Clamped {fixed} out-of-range predictions to 0.")

# Sanity vs sample_submission
sample_sub_path = os.path.join(DATA_ROOT, "sample_submission.csv")
if os.path.exists(sample_sub_path):
    samp = pd.read_csv(sample_sub_path)
    assert set(samp["id"]) == set(submission["id"]), "Submission ids do not match sample_submission ids."
    submission = submission.set_index("id").loc[samp["id"]].reset_index()  # match order
    assert list(submission.columns) == ["id", "answer"]

out_path = os.path.join(OUTPUT_DIR, "submission.csv")
submission.to_csv(out_path, index=False)
print(f"Wrote {out_path}  ({len(submission)} rows)")
print(submission.head())
print("\nAnswer distribution:", submission["answer"].value_counts().sort_index().to_dict())

## 12. (Optional) Save the LoRA adapter

Useful if you want to re-run inference later without retraining. Skip if you don't need it.

In [ ]:
if DO_TRAIN:
    adapter_dir = os.path.join(OUTPUT_DIR, "lora_adapter")
    model.save_pretrained(adapter_dir)
    print(f"Saved LoRA adapter to {adapter_dir}")
else:
    print("No adapter to save (DO_TRAIN=False).")

## Notes & tuning tips

- **Trainable-parameter budget.** With `LORA_R=8` on `q,k,v,o` of the language model attention only, you'll see well under 5M trainable params. The cell in §5 hard-asserts this so you can't accidentally exceed it.
- **Rules-compliant offline run.** When the grader runs this notebook offline, the `pip install` cells become no-ops if the right versions are pre-installed in the Kaggle image. The model itself is downloaded from HuggingFace at training time and cached; for a strictly-offline rerun, attach the model as a Kaggle dataset and point `MODEL_ID` at that local path.
- **If you're memory-constrained.** Drop `MAX_IMG_SIZE` to 320, keep `BATCH_SIZE=1`, raise `GRAD_ACCUM_STEPS` to 16. SmolVLM-500M with `bfloat16/float16` and image size 384 fits on a T4 with this config.
- **If you have time to spare.** Increase `NUM_EPOCHS` to 2–3; the LoRA adapter is small enough to keep learning from this dataset without overfitting catastrophically. Watch the validation accuracy in §10.
- **Zero-training baseline.** Set `DO_TRAIN = False` in §1 to skip fine-tuning entirely. SmolVLM-Instruct already follows MCQA prompts surprisingly well, so this gives you a useful sanity check before committing GPU time.